# BTS — Etapa B2: regresión logística original (SageMaker, checkpoints de etapa A)

**Continuación de B1, sin volver a ejecutar Bronze, Silver ni la ingeniería de variables.** Reutiliza exactamente los Parquet `features_seed42_h240_chunk50000` guardados en Etapa A. Conserva la definición y parámetros originales de la regresión logística: **1.000.000 de filas de entrenamiento**, 1.000.000 en validación y 1.000.000 en prueba.

- La regresión logística era la **línea base del notebook original**, que usa solamente el primer millón de filas del train original (`.iloc[:N_LOGREG]`). No se está reduciendo una muestra que originalmente fuera de 4 millones para este algoritmo.
- Para evitar mantener varios modelos y matrices en memoria, se entrena aquí por separado, guarda el modelo y luego calcula validación y prueba **por separado**.
- Verifica que `y` y su **orden** coincidan con B1 antes de guardar las probabilidades. Así las comparaciones posteriores serán pareadas.
- La selección definitiva del ganador se hará más adelante y debe basarse en validación; la prueba se reserva para la comparación final.

**Ejecutar desde `/home/ec2-user/SageMaker/src/`.** No requiere archivo DuckDB ni descarga Silver.

In [1]:
%pip install -q scikit-learn pandas numpy pyarrow joblib boto3


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 0. Rutas y configuración originales

In [2]:
from pathlib import Path
import os, gc, json, time, shutil, sys
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import boto3, joblib
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
assert ROOT.name == "src", f"Abre el notebook desde /home/ec2-user/SageMaker/src/; ahora estás en {ROOT}"
PARTS_BASE = Path("../data/checkpoints/features_seed42_h240_chunk50000")
B1_DIR = Path("../outputs/outputs_modeling/checkpoints/lightgbm")
OUT_DIR = Path("../outputs/outputs_modeling/checkpoints/logreg")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = "DepDel15"
SEED = 42
N_TRAIN, N_EVAL, N_LOGREG = 4_000_000, 1_000_000, 1_000_000
COST_FP, COST_FN = 1.0, 3.0
BUCKET = "bts-flights-analysis"
B1_PREFIX = "modeling/checkpoints/lightgbm-original-seed42-h240"
S3_PREFIX = "modeling/checkpoints/logistic-regression-original-seed42-h240"
s3 = boto3.client("s3")
print("Checkpoint A:", PARTS_BASE.resolve())
print("Salida B2:", OUT_DIR.resolve())
print("Disco libre:", round(shutil.disk_usage(ROOT).free/1024**3, 2), "GiB")

Checkpoint A: /home/ec2-user/SageMaker/data/checkpoints/features_seed42_h240_chunk50000
Salida B2: /home/ec2-user/SageMaker/outputs/outputs_modeling/checkpoints/logreg
Disco libre: 25.6 GiB


## 1. Validación rápida de los Parquet (sin cargar sus filas en RAM)

In [3]:
PARTS = {}
for split, expected in {"train": N_TRAIN, "val": N_EVAL, "test": N_EVAL}.items():
    folder = PARTS_BASE / split
    files = sorted(folder.glob("part_*.parquet"))
    assert (folder / "_SUCCESS").exists(), f"Falta _SUCCESS en {folder}"
    assert files, f"No hay bloques Parquet en {folder}"
    rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
    assert rows == expected, f"{split}: {rows:,} filas, se esperaban {expected:,}"
    PARTS[split] = files
    print(f"✔ {split}: {len(files)} archivos, {rows:,} filas")

✔ train: 80 archivos, 4,000,000 filas
✔ val: 20 archivos, 1,000,000 filas
✔ test: 20 archivos, 1,000,000 filas


## 2. Variables y pipeline idénticos al del notebook original

Se usan las columnas del apartado **7.1 Regresión logística**. Incluyen `dep_hour`, que la rama LightGBM no necesita, pero que **sí está en los Parquet generados por la etapa A**.

In [4]:
HIST_KEYS = ["origin", "route", "airline", "tail", "origin_hr"]
HIST_COLS = [f"{k}_rate_{s}" for k in HIST_KEYS for s in ("all", "30d")]
SCHED_COLS = ["origin_dep_load", "origin_day_load", "dest_arr_load", "tail_leg_num", "turnaround_min"]
STATUS_COLS = ["prev_leg_delayed", "tail_delayed_known", "tail_known_legs", "origin_known_today", "origin_delay_rate_today"]
LR_NUM = (["Distance", "CRSElapsedTime", "dep_min", "arr_min", "days_to_holiday"]
          + HIST_COLS + SCHED_COLS + STATUS_COLS)
LR_CAT = ["Reporting_Airline", "Origin", "Dest", "DepTimeBlk", "Month", "DayOfWeek", "dep_hour", "is_weekend"]
LR_COLS = LR_NUM + LR_CAT
COLS_TO_READ = LR_COLS + [TARGET]
schema_cols = set(pq.ParquetFile(PARTS["train"][0]).schema_arrow.names)
missing = [x for x in COLS_TO_READ if x not in schema_cols]
assert not missing, f"Faltan columnas originales de regresión logística: {missing}"

def make_lr():
    return Pipeline([
        ("pre", ColumnTransformer([
            ("num", Pipeline([
                ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                ("sc", StandardScaler())]), LR_NUM),
            ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100), LR_CAT),
        ])),
        ("clf", LogisticRegression(max_iter=300, C=1.0)),
    ])

print("Columnas de LR:", len(LR_COLS), "(numéricas", len(LR_NUM), ", categóricas", len(LR_CAT), ")")

Columnas de LR: 33 (numéricas 25 , categóricas 8 )


## 3. Cargar únicamente lo que exige cada etapa

Para entrenamiento se lee **el primer millón de filas del train** (como `.iloc[:N_LOGREG]` en el notebook original). Los conjuntos completos de validación y prueba se cargan por separado y se liberan después de predecir.

In [5]:
def load_split(split, max_rows=None):
    t0 = time.time()
    parts = []
    total = 0
    for i, path in enumerate(PARTS[split], 1):
        if max_rows is not None and total >= max_rows:
            break
        frame = pd.read_parquet(path, columns=COLS_TO_READ)
        if max_rows is not None and total + len(frame) > max_rows:
            frame = frame.iloc[:max_rows-total].copy()
        parts.append(frame)
        total += len(frame)
        if i % 10 == 0:
            print(f"  {split}: bloques {i}/{len(PARTS[split])} | filas {total:,}")
    result = pd.concat(parts, ignore_index=True)
    del parts
    gc.collect()
    if max_rows is not None:
        assert len(result) == max_rows, f"Solo se cargaron {len(result):,} de {max_rows:,}"
    print(f"✔ {split}: {len(result):,} filas, {result.memory_usage(deep=True).sum()/1024**3:.2f} GiB en pandas, {time.time()-t0:.0f} s")
    return result

def check_b1_labels(split, y):
    """Exige que las etiquetas coincidan fila por fila con B1 (local o respaldado en S3)."""
    b1_path = B1_DIR / f"{split}_predictions.npz"
    if not b1_path.exists():
        B1_DIR.mkdir(parents=True, exist_ok=True)
        s3.download_file(BUCKET, f"{B1_PREFIX}/{split}_predictions.npz", str(b1_path))
        print("Recuperado checkpoint B1 desde S3:", b1_path.name)
    with np.load(b1_path) as b1:
        y_b1 = b1["y"]
        assert y.shape == y_b1.shape and np.array_equal(y, y_b1), (
            f"No coinciden etiquetas/orden de {split} frente a B1; detener para evitar comparaciones inválidas"
        )
    print(f"✔ Etiquetas y orden de {split} idénticos a B1")

## 4. Entrenar regresión logística (primer millón original) y guardar modelo

In [6]:
train = load_split("train", max_rows=N_LOGREG)
y_train = train[TARGET].to_numpy(dtype=np.uint8)
t0 = time.time()
model = make_lr()
with threadpool_limits(limits=4):
    model.fit(train[LR_COLS], y_train)
training_seconds = round(time.time() - t0, 2)
print("✔ Regresión logística entrenada en", training_seconds, "s")
joblib.dump(model, OUT_DIR / "model.joblib", compress=3)
run_config = {
    "model": "Regresión logística", "seed": SEED,
    "n_logreg_original": N_LOGREG, "train_rows_available": N_TRAIN,
    "val_rows": N_EVAL, "test_rows": N_EVAL,
    "train_end": "2024-12-31", "val_end": "2025-08-31",
    "prediction_horizon_min": 240, "training_seconds": training_seconds,
    "pipeline": "SimpleImputer(median, add_indicator) + StandardScaler; OneHotEncoder(min_frequency=100); LogisticRegression(max_iter=300,C=1.0)",
    "lr_num": LR_NUM, "lr_cat": LR_CAT,
    "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
(OUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False))
del train, y_train
gc.collect()
print("✔ Modelo completo, incluido preprocesamiento, guardado:", (OUT_DIR / "model.joblib").resolve())

  train: bloques 10/80 | filas 500,000
  train: bloques 20/80 | filas 1,000,000
✔ train: 1,000,000 filas, 0.44 GiB en pandas, 1 s
✔ Regresión logística entrenada en 41.82 s
✔ Modelo completo, incluido preprocesamiento, guardado: /home/ec2-user/SageMaker/outputs/outputs_modeling/checkpoints/logreg/model.joblib


## 5. Predecir validación y liberar memoria

In [7]:
val = load_split("val")
y_val = val[TARGET].to_numpy(dtype=np.uint8)
check_b1_labels("val", y_val)
with threadpool_limits(limits=4):
    p_val = model.predict_proba(val[LR_COLS])[:, 1]
np.savez_compressed(OUT_DIR / "val_predictions.npz", y=y_val, p=p_val)
print("✔ Validación: probabilidad media", round(float(p_val.mean()), 5))
del val
gc.collect()

  val: bloques 10/20 | filas 500,000
  val: bloques 20/20 | filas 1,000,000
✔ val: 1,000,000 filas, 0.44 GiB en pandas, 1 s
✔ Etiquetas y orden de val idénticos a B1
✔ Validación: probabilidad media 0.21837


57

## 6. Predecir prueba y liberar memoria

In [8]:
test = load_split("test")
y_test = test[TARGET].to_numpy(dtype=np.uint8)
check_b1_labels("test", y_test)
with threadpool_limits(limits=4):
    p_test = model.predict_proba(test[LR_COLS])[:, 1]
np.savez_compressed(OUT_DIR / "test_predictions.npz", y=y_test, p=p_test)
print("✔ Prueba: probabilidad media", round(float(p_test.mean()), 5))
del test, model
gc.collect()

  test: bloques 10/20 | filas 500,000
  test: bloques 20/20 | filas 1,000,000
✔ test: 1,000,000 filas, 0.44 GiB en pandas, 1 s
✔ Etiquetas y orden de test idénticos a B1
✔ Prueba: probabilidad media 0.22252


57

## 7. Métricas y umbral por costo sobre validación

Calcula el umbral únicamente con **validación**. Guarda métricas descriptivas de prueba; la elección entre modelos se realizará cuando estén listos los cuatro checkpoints.

In [10]:
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss

sys.path.insert(0, str(Path("utils").resolve()))
import model_eval as me
threshold_val = float(me.best_cost_threshold(y_val, p_val, c_fp=COST_FP, c_fn=COST_FN))

def numpy_to_json(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Tipo no serializable: {type(obj)}")

metrics = {
    "val": me.evaluate_model("Regresión logística", y_val, p_val, threshold_val, c_fp=COST_FP, c_fn=COST_FN),
    "test": me.evaluate_model("Regresión logística", y_test, p_test, threshold_val, c_fp=COST_FP, c_fn=COST_FN),
    "threshold_from_validation": threshold_val,
    "threshold_theoretical": float(COST_FP/(COST_FP + COST_FN)),
}
(OUT_DIR / "metrics.json").write_text(
    json.dumps(
        metrics,
        indent=2,
        ensure_ascii=False,
        default=numpy_to_json
    )
)
print(json.dumps({k: {metric: v for metric, v in stats.items() if metric in ("pr_auc", "roc_auc", "brier", "log_loss", "precision", "recall", "f1", "costo_x1000")}
                  for k, stats in metrics.items() if k in ("val", "test")}, indent=2, ensure_ascii=False, default=numpy_to_json))
assert all((OUT_DIR / f).exists() for f in ("model.joblib", "run_config.json", "val_predictions.npz", "test_predictions.npz", "metrics.json"))
print("✔ B2 finalizado: modelo, etiquetas, probabilidades y métricas guardados")
del y_val, y_test, p_val, p_test
gc.collect()

{
  "val": {
    "roc_auc": 0.7293954485918215,
    "pr_auc": 0.4707529702760106,
    "precision": 0.4187125419951716,
    "recall": 0.556828795172067,
    "f1": 0.4779934819771464,
    "costo_x1000": 468.245,
    "brier": 0.14942786402141267,
    "log_loss": 0.4673068056031199
  },
  "test": {
    "roc_auc": 0.7344348004090197,
    "pr_auc": 0.4890630755844013,
    "precision": 0.42618294787262734,
    "recall": 0.5720317359236174,
    "f1": 0.4884523849428485,
    "costo_x1000": 470.93600000000004,
    "brier": 0.15093138189876104,
    "log_loss": 0.47088613318622874
  }
}
✔ B2 finalizado: modelo, etiquetas, probabilidades y métricas guardados


309

## 8. Respaldo de B2 en S3 (ejecutar después de completar B2)

Guarda **este modelo candidato**. No es necesariamente el ganador definitivo ni despliega un endpoint. Sobrescribe únicamente su prefijo de checkpoint, nunca `models/flight-delay/`.

In [11]:
files = ["model.joblib", "run_config.json", "val_predictions.npz", "test_predictions.npz", "metrics.json"]
assert all((OUT_DIR / f).exists() for f in files), "B2 incompleto; no subir"
for file in files:
    key = f"{S3_PREFIX}/{file}"
    s3.upload_file(str(OUT_DIR / file), BUCKET, key)
    print("Subido:", f"s3://{BUCKET}/{key}")
print("✔ Checkpoint B2 respaldado en S3")

Subido: s3://bts-flights-analysis/modeling/checkpoints/logistic-regression-original-seed42-h240/model.joblib
Subido: s3://bts-flights-analysis/modeling/checkpoints/logistic-regression-original-seed42-h240/run_config.json
Subido: s3://bts-flights-analysis/modeling/checkpoints/logistic-regression-original-seed42-h240/val_predictions.npz
Subido: s3://bts-flights-analysis/modeling/checkpoints/logistic-regression-original-seed42-h240/test_predictions.npz
Subido: s3://bts-flights-analysis/modeling/checkpoints/logistic-regression-original-seed42-h240/metrics.json
✔ Checkpoint B2 respaldado en S3


## Próximo paso: B3 (XGBoost)

Después de comprobar que los cinco archivos B2 están en S3, reiniciar el kernel y ejecutar XGBoost en un notebook independiente, reutilizando **los mismos 120 Parquet de Etapa A**. Después seguiremos con K-means y la comparación final, sin repetir la ingeniería histórica.